# Notebook 3 — Win-Plan Recommendation Agent for Field Reps
**Aurevia flu vaccines · uses interactions + call notes + the win-probability model from Notebook 2**

**What it does:** for any 2025-26 opportunity, an LLM agent (Claude Sonnet 4.5 on Amazon Bedrock) gathers evidence with four tools, then returns a short, evidence-backed **win plan** for the rep.

```
Question  ──►  resolve opportunity  ──►  Claude agent  ──┬─ get_deal_brief          (model probability, drivers, price what-if)
                                                          ├─ get_recent_interactions (contact cadence, issues, committee)
                                                          ├─ search_account_notes    (embedding search over call notes)
                                                          ├─ get_account_history     (this account's past deals: outcome, price gap, rounds)
                                                          ├─ get_portfolio_view      (the account's other open 2025-26 deals)
                                                          ├─ get_similar_past_deals  (win rate + typical negotiation of comparable past deals)
                                                          └─ submit_recommendation   (validated JSON plan)  ──►  QA checks  ──►  one-page brief
```

**Design choices (kept deliberately simple)**
* **Retrieval, not bulk LLM processing:** call notes are embedded once (Titan Embeddings; 535 unique texts → cheap) and only the *relevant* notes for an account are shown to the agent.
* **Grounded:** the agent may only use tool outputs, must cite note ids, and may only pick actions from an approved **playbook** (edit it to Aurevia's real plays).
* **Advisory only:** price guidance stays inside the model's −5 % … +5 % what-if range; a person approves every price. Contact names are masked before any text reaches the LLM.
* **Safe fallback:** `DRY_RUN = True` runs everything offline with a rule-based stand-in (no AWS needed) — useful for testing.

**Run order:** Notebooks 1 and 2 first (this needs `data/processed/model_table.csv` and `predictions_2025_26.csv`). Then *Restart Kernel and Run All*, and use the **last cell** to ask your question.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import re, json, time, math, pickle, difflib
import numpy as np, pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)

RAW_DIR, INTERMEDIATE_DIR, PROCESSED_DIR = "../data/raw/", "../data/intermediate/", "../data/processed/"

# ---------------- AWS Bedrock settings ----------------
REGION          = "us-east-1"
LLM_MODEL_ID    = "arn:aws:bedrock:us-east-1:025540956822:application-inference-profile/uwuvuaamy8n4"   # Claude Sonnet 4.5
EMBED_MODEL_ID  = "amazon.titan-embed-text-v2:0"
DRY_RUN         = True      # True = offline test mode (rule-based stand-in for Claude + TF-IDF instead of Titan)
WINDOW_DAYS     = 365        # how far back the agent may look

## 1. Load data
Notes are masked (contact names removed) **before** indexing, so no personal names ever reach the LLM.

In [2]:
interactions = pd.read_csv(RAW_DIR + "interactions.csv", parse_dates=["interaction_date"])
notes_raw    = pd.read_csv(RAW_DIR + "interaction_notes.csv", parse_dates=["note_date"])
customers    = pd.read_csv(RAW_DIR + "customers.csv")
preds        = pd.read_csv(PROCESSED_DIR + "predictions_2025_26.csv")
mt           = pd.read_csv(PROCESSED_DIR + "model_table.csv", parse_dates=["offer_date"])

def mask_contacts(t):
    t = re.sub(r"\bDr\.\s+[A-Z][a-z]+", "[CONTACT]", t)
    t = re.sub(r"\b(Michelle|Sarah)\b", "[CONTACT]", t)
    return re.sub(r"\s+", " ", t).strip()

interactions = interactions.drop_duplicates("interaction_id")
notes = (notes_raw.drop_duplicates("note_id")
         .assign(text=lambda d: d["note_text"].fillna("").map(mask_contacts))
         .merge(interactions[["interaction_id", "interaction_date", "interaction_type", "product_focus", "rep_id", "issue_flag_raised"]], on="interaction_id", how="left"))
AS_OF = interactions["interaction_date"].max()          # "today" inside this dataset: the latest recorded activity

# one row per open 2025-26 deal: model output + account/price context
score = mt[mt.dataset == "score"][["opportunity_id", "customer_id", "product_id", "offer_date", "region", "relationship_years", "has_buying_group",
                                   "comp_family", "sim_comp_price", "sim_list_price", "is_incumbent"]]
deals = (preds.merge(score, on=["opportunity_id", "customer_id"])
              .merge(customers[["customer_id", "bed_count", "annual_flu_doses_est"]], on="customer_id", how="left")
              .set_index("opportunity_id", drop=False))
rep_of = interactions.groupby("customer_id")["rep_id"].agg(lambda s: s.mode().iloc[0])      # main rep per account
deals["rep_id"] = deals["customer_id"].map(rep_of)
history = mt[mt.dataset == "history"].copy()                                              # for "similar past deals"
contracts_hist = pd.read_csv(RAW_DIR + "contracts_history.csv", parse_dates=["offer_date", "outcome_date"])
history = history.merge(contracts_hist[["opportunity_id", "outcome", "final_negotiated_price_per_dose", "num_negotiation_rounds", "competitor_product"]], on="opportunity_id", how="left")
history["won"] = history.target.astype(int)
history["concession_pct"] = (1 - history.final_negotiated_price_per_dose / history.sim_offer_price) * 100     # only defined for WON deals
print(f"{len(deals)} open deals | {len(notes)} notes | {notes.text.nunique()} unique texts | data as of {AS_OF:%Y-%m-%d} | model {deals.model_version.iloc[0]}")

430 open deals | 3502 notes | 535 unique texts | data as of 2025-12-25 | model logistic_L1_20261117


## 1b. Which structured data is actually worth giving the agent?
Rather than add every table, each candidate is tested on the labelled history. Past results are legitimate coaching *context* even where they are not *predictive*.

In [3]:
ch = contracts_hist.assign(won=lambda d: (d.outcome == "WON").astype(int), sy=lambda d: d.season.str[:4].astype(int),
                           conc=lambda d: 1 - d.final_negotiated_price_per_dose / d.initial_offered_price_per_dose)
by_c = {k: g for k, g in ch.groupby("customer_id")}
prior_wr, now, c_prev, c_now = [], [], [], []
for r in ch.itertuples():
    prev = by_c[r.customer_id][by_c[r.customer_id].sy < r.sy]
    if len(prev): prior_wr.append(prev.won.mean()); now.append(r.won)
    pw = prev[prev.won == 1]
    if r.won == 1 and len(pw): c_prev.append(pw.conc.mean()); c_now.append(r.conc)
piv = ch.pivot_table(index=["customer_id", "season"], columns="product_id", values="won")
pairs = np.array([r.dropna().values[:2] for _, r in piv.iterrows() if r.notna().sum() >= 2])
CROSS_PRODUCT_CORR = float(np.corrcoef(pairs[:, 0], pairs[:, 1])[0, 1])
evid = pd.DataFrame([
 ("Account's prior-season win/loss record",        f"corr with next outcome = {np.corrcoef(prior_wr, now)[0, 1]:+.2f} (n={len(now)})",                    "USED as context (tool: get_account_history)"),
 ("Account's own past concession size",            f"persistence corr = {np.corrcoef(c_prev, c_now)[0, 1]:+.2f} (n={len(c_now)}); won-deal concession ~{ch.conc.mean() * 100:.1f}% everywhere", "NOT account-specific; only a generic benchmark"),
 ("Account's other open products (same season)",   f"cross-product outcome corr = {CROSS_PRODUCT_CORR:+.2f} (n={len(pairs)})",                           "USED as context for bundling (tool: get_portfolio_view); no statistical link"),
 ("Comparable past deals (incumbency, type, price)", "base-rate evidence",                                                                           "USED (tool: get_similar_past_deals)"),
 ("Model probability, drivers, price what-if",     "Notebook 2 output",                                                                               "USED (tool: get_deal_brief)"),
 ("Interaction cadence / issue flags",             "structured engagement",                                                                           "USED (tool: get_recent_interactions)"),
 ("negotiation_rounds / negotiation_decks tables", "do not reconcile with contracts; no 2025-26 coverage",                                          "EXCLUDED"),
], columns=["structured data", "evidence", "decision"])
evid

,structured data,evidence,decision
0,Account's prior-season win/loss record,corr with next outcome = +0.24 (n=786),USED as context (tool: get_account_history)
1,Account's own past concession size,persistence corr = -0.08 (n=188); won-deal con...,NOT account-specific; only a generic benchmark
2,Account's other open products (same season),cross-product outcome corr = -0.00 (n=188),USED as context for bundling (tool: get_portfo...
3,"Comparable past deals (incumbency, type, price)",base-rate evidence,USED (tool: get_similar_past_deals)
4,"Model probability, drivers, price what-if",Notebook 2 output,USED (tool: get_deal_brief)
5,Interaction cadence / issue flags,structured engagement,USED (tool: get_recent_interactions)
6,negotiation_rounds / negotiation_decks tables,do not reconcile with contracts; no 2025-26 co...,EXCLUDED


## 2. Embeddings & note search
Titan Text Embeddings V2 (512-d, normalised). Only **unique** note texts are embedded, then cached to `data/intermediate/`. The same class falls back to TF-IDF in `DRY_RUN`.

Search = cosine similarity to the question **+ small recency bonus** (half-life 90 days), restricted to the account, the look-back window, and notes about this product or the whole portfolio.

In [4]:
_bedrock = {}
def bedrock():
    # lazily create the runtime client (boto3 is preinstalled on SageMaker); credentials come from the notebook's IAM role
    if "c" not in _bedrock:
        import boto3
        from botocore.config import Config
        _bedrock["c"] = boto3.client("bedrock-runtime", region_name=REGION, config=Config(read_timeout=120, retries={"max_attempts": 4, "mode": "standard"}))
    return _bedrock["c"]

def titan_embed(text):
    r = bedrock().invoke_model(modelId=EMBED_MODEL_ID, contentType="application/json", accept="application/json",
                               body=json.dumps({"inputText": text, "dimensions": 512, "normalize": True}))
    return np.array(json.loads(r["body"].read())["embedding"], dtype="float32")

class NoteIndex:
    def __init__(self, notes_df, dry_run):
        self.df, self.dry = notes_df.reset_index(drop=True), dry_run
        self.texts = sorted(self.df["text"].unique()); self.row = {t: i for i, t in enumerate(self.texts)}
        if dry_run:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(self.texts)
            self.E = self.vec.transform(self.texts).toarray().astype("float32")
        else:
            path = INTERMEDIATE_DIR + "note_embeddings.pkl"
            try: cache = pickle.load(open(path, "rb"))
            except Exception: cache = {}
            todo = [t for t in self.texts if t not in cache]
            for k, t in enumerate(todo):
                cache[t] = titan_embed(t)
                if (k + 1) % 100 == 0: print(f"  embedded {k + 1}/{len(todo)}")
            pickle.dump(cache, open(path, "wb"))
            self.E = np.vstack([cache[t] for t in self.texts])
        self.df["erow"] = self.df["text"].map(self.row)

    def encode_query(self, q):
        if self.dry:
            v = self.vec.transform([q]).toarray()[0].astype("float32")
            return v / (np.linalg.norm(v) + 1e-9)
        return titan_embed(q)

    def search(self, customer_id, query, as_of, product_id, k=5, window_days=WINDOW_DAYS):
        d = self.df[(self.df.customer_id == customer_id) & (self.df.interaction_date <= as_of) &
                    (self.df.interaction_date >= as_of - pd.Timedelta(days=window_days)) &
                    (self.df.product_focus.isin(["PORTFOLIO", product_id]))].copy()
        if d.empty: return []
        sim = self.E[d["erow"].values] @ self.encode_query(query)
        recency = np.exp(-((as_of - d["interaction_date"]).dt.days.values) / 90.0)
        d["score"] = sim + 0.10 * recency
        d = d.sort_values("score", ascending=False).drop_duplicates("text").head(k)
        return [dict(note_id=r.note_id, date=f"{r.interaction_date:%Y-%m-%d}", channel=r.interaction_type, product=r.product_focus, text=r.text, relevance=round(float(r.score), 3))
                for r in d.itertuples()]

t0 = time.time(); INDEX = NoteIndex(notes, DRY_RUN)
print(f"note index ready ({'TF-IDF, dry run' if DRY_RUN else 'Titan embeddings'}) in {time.time() - t0:.0f}s | embedding matrix {INDEX.E.shape}")

note index ready (TF-IDF, dry run) in 3s | embedding matrix (535, 1274)


In [5]:
# quick retrieval sanity check
cid = deals.customer_id.iloc[0]; pid = deals.product_id.iloc[0]
for h in INDEX.search(cid, "competitor threatens to switch or re-tender", AS_OF, pid, k=3): print(h["date"], h["note_id"], "|", h["text"][:120])

2025-05-19 NOTE0002629 | Heard through the [CONTACT] channel that AeroFlu has been visiting weekly. They're pushing hard on their AeroFlu. We nee
2025-07-25 NOTE0002631 | Great meeting with [CONTACT] in supply chain. Their team highlighted our portfolio's efficacy in their high-risk populat


## 3. Tools the agent can call
Every tool returns plain JSON. Only information up to `AS_OF` is available, and the model probability is clearly labelled as an **offer-time** estimate.

In [6]:
def jsonable(o):
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (float, np.floating)): return None if (math.isnan(o) or math.isinf(o)) else round(float(o), 4)
    if isinstance(o, (pd.Timestamp,)): return o.strftime("%Y-%m-%d")
    return o

def get_deal_brief(opportunity_id):
    d = deals.loc[opportunity_id]
    what_if = {k: d[f"p_{k}"] for k in ["-5.0%", "-2.5%", "+0.0%", "+2.5%", "+5.0%"]}
    return jsonable({
        "opportunity_id": opportunity_id, "customer": d.customer_name, "customer_type": d.customer_type, "region": d.region, "product": d.product_name,
        "incumbent_supplier": d.incumbent_supplier, "main_competitor_family": d.comp_family, "offer_date": d.offer_date,
        "our_price_per_dose": d.sim_offer_price, "competitor_price_per_dose": d.sim_comp_price, "price_vs_competitor_pct": d.price_vs_comp_pct,
        "account": {"beds": d.bed_count, "est_annual_doses": d.annual_flu_doses_est, "relationship_years": d.relationship_years, "in_buying_group": bool(d.has_buying_group)},
        "model_estimate_at_offer_time": {"win_probability": d.win_probability, "risk_tier": str(d.risk_tier), "drivers_for_win": d.drivers_for_win,
                                         "drivers_against_win": d.drivers_against_win, "what_if_win_probability_by_price_change": what_if,
                                         "gain_from_5pct_price_cut": d.gain_from_5pct_cut},
        "note": "Probability uses information known at offer time only; newer notes may change the picture. What-if is historical association, not proven cause."})

def get_recent_interactions(opportunity_id, n=8):
    d = deals.loc[opportunity_id]; x = interactions[(interactions.customer_id == d.customer_id) & (interactions.interaction_date <= AS_OF) &
                                                    (interactions.interaction_date >= AS_OF - pd.Timedelta(days=WINDOW_DAYS))].sort_values("interaction_date")
    if x.empty: return {"n_interactions_12m": 0, "message": "No interactions on record in the last 12 months"}
    last90 = x[x.interaction_date > AS_OF - pd.Timedelta(days=90)]; prev90 = x[(x.interaction_date <= AS_OF - pd.Timedelta(days=90)) & (x.interaction_date > AS_OF - pd.Timedelta(days=180))]
    return jsonable({"n_interactions_12m": len(x), "n_last_90d": len(last90), "n_prior_90d": len(prev90), "days_since_last_contact": (AS_OF - x.interaction_date.max()).days,
                     "issue_flags_12m": int(x.issue_flag_raised.sum()), "clinical_committee_presentations_12m": int((x.interaction_type == "clinical_committee_presentation").sum()),
                     "distinct_reps": int(x.rep_id.nunique()),
                     "latest": [dict(date=r.interaction_date, type=r.interaction_type, product=r.product_focus, minutes=r.duration_minutes, issue_flag=bool(r.issue_flag_raised)) for r in x.tail(n).itertuples()]})

def search_account_notes(opportunity_id, query, k=5):
    d = deals.loc[opportunity_id]; hits = INDEX.search(d.customer_id, query, AS_OF, d.product_id, k=int(k))
    return {"query": query, "n_results": len(hits), "notes": hits} if hits else {"query": query, "n_results": 0, "message": "No matching notes"}

def get_similar_past_deals(opportunity_id):
    d = deals.loc[opportunity_id]; h = history
    for label, cond in [("same incumbency, customer type and product, price within +/-3 pts", (h.is_incumbent == d.is_incumbent) & (h.customer_type == d.customer_type) & (h.product_id == d.product_id)),
                        ("same incumbency and customer type, price within +/-3 pts", (h.is_incumbent == d.is_incumbent) & (h.customer_type == d.customer_type)),
                        ("same incumbency, price within +/-3 pts", (h.is_incumbent == d.is_incumbent))]:
        s = h[cond & ((h.price_vs_comp_pct - d.price_vs_comp_pct).abs() <= 3)]
        if len(s) >= 15: break
    won = s[s.won == 1]
    return jsonable({"definition": label, "n_past_deals": len(s), "win_rate": s.target.mean() if len(s) else None, "overall_history_win_rate": h.target.mean(),
                     "typical_when_won": {"avg_negotiation_rounds": won.num_negotiation_rounds.mean(), "avg_concession_pct_off_initial_offer": won.concession_pct.mean(),
                                          "caution": "benchmark for comparable deals only; an account's own past concession does not predict its next one"}})

def get_account_history(opportunity_id):
    d = deals.loc[opportunity_id]; h = history[history.customer_id == d.customer_id].sort_values(["season", "product_name"])
    if h.empty: return {"n_past_deals": 0, "message": "No past deals on record (new account)."}
    rows = [dict(season=r.season, product=r.product_name, outcome=r.outcome, our_price_vs_competitor_pct=r.price_vs_comp_pct, incumbent_then=r.incumbent_supplier,
                 competitor=r.competitor_product, rounds=r.num_negotiation_rounds, concession_pct=(r.concession_pct if r.won else None)) for r in h.itertuples()]
    same = [x for x in rows if x["product"] == d.product_name]
    return jsonable({"n_past_deals": len(h), "n_won": int(h.won.sum()), "same_product_record": same, "past_deals_latest": rows[-8:]})

def get_portfolio_view(opportunity_id):
    d = deals.loc[opportunity_id]; o = deals[(deals.customer_id == d.customer_id) & (deals.opportunity_id != opportunity_id)]
    return jsonable({"n_other_open_deals": len(o),
                     "other_open_deals": [dict(opportunity_id=r.opportunity_id, product=r.product_name, win_probability=r.win_probability, risk_tier=str(r.risk_tier),
                                               price_vs_competitor_pct=r.price_vs_comp_pct, gain_from_5pct_cut=r.gain_from_5pct_cut) for r in o.itertuples()],
                     "note": f"Outcomes of different products at the same account were not statistically linked in history (correlation {CROSS_PRODUCT_CORR:+.2f}); a bundle is a commercial lever, not a proven predictor."})

TOOLS = {"get_deal_brief": get_deal_brief, "get_recent_interactions": get_recent_interactions, "search_account_notes": search_account_notes,
         "get_account_history": get_account_history, "get_portfolio_view": get_portfolio_view, "get_similar_past_deals": get_similar_past_deals}
oid = deals.index[0]; print(json.dumps(get_deal_brief(oid), indent=1)[:700], "\n...")
print(get_similar_past_deals(oid)); print(get_account_history(oid)["n_past_deals"], "past deals;", get_portfolio_view(oid)["n_other_open_deals"], "other open deals")

{
 "opportunity_id": "OPP001113",
 "customer": "Harbor Riverside Medical Center",
 "customer_type": "Health System",
 "region": "Northeast",
 "product": "Vaxora Standard",
 "incumbent_supplier": "AeroFlu",
 "main_competitor_family": "AeroFlu",
 "offer_date": "2025-03-17",
 "our_price_per_dose": 19.08,
 "competitor_price_per_dose": 19.08,
 "price_vs_competitor_pct": 0.0,
 "account": {
  "beds": 1562,
  "est_annual_doses": 7497,
  "relationship_years": 5,
  "in_buying_group": false
 },
 "model_estimate_at_offer_time": {
  "win_probability": 0.292,
  "risk_tier": "Contested",
  "drivers_for_win": "has_buying_group (+0.09); price_vs_comp_pct (+0.04)",
  "drivers_against_win": "is_incumbent (-0.2 
...
{'definition': 'same incumbency, customer type and product, price within +/-3 pts', 'n_past_deals': 41, 'win_rate': 0.2683, 'overall_history_win_rate': 0.3498, 'typical_when_won': {'avg_negotiation_rounds': 3.0909, 'avg_concession_pct_off_initial_offer': 4.7685, 'caution': "benchmark for compa

## 4. Playbook and output schema
**Edit `PLAYBOOK` to Aurevia's real, approved plays** — the agent can only recommend from this list. The submit tool enforces the JSON structure, so the output is always machine-readable.

In [7]:
PLAYBOOK = {   # EXAMPLE plays - replace with the company's approved playbook
 "EXEC_SPONSOR":        "Executive-to-executive outreach to the decision makers (CMO / CPO / pharmacy director) for accounts under real competitive threat.",
 "PRICE_VALUE_DEFENCE": "Answer price pressure with a total-value case (supply reliability, service, clinical data) before moving price.",
 "TARGETED_CONCESSION": "Small, conditional price move (within the model's +/-5% range, manager-approved) tied to volume or multi-product commitment. Only where the model shows price-responsiveness.",
 "CLINICAL_EDUCATION":  "Schedule a clinical-committee session / data review (e.g. immunogenicity, high-risk populations) to reinforce clinical preference.",
 "SERVICE_RECOVERY":    "Resolve open service / supply / billing issues with a named owner and dated commitments before the next negotiation round.",
 "PORTFOLIO_BUNDLE":    "Offer a cross-product package (Vaxora HD + Vaxora Standard + Immunexa) instead of discounting a single product.",
 "COMPETITOR_COUNTER":  "Prepare a factual comparison against the named competitor and address its specific claims or offers.",
 "ENGAGEMENT_BOOST":    "Raise contact cadence and add stakeholders (pharmacy, clinical committee, procurement) - multi-thread the account.",
 "EXPANSION_PLAY":      "Account signals growth: propose expanded volume / additional sites on current terms.",
 "HOLD_AND_PROTECT":    "Strong position: hold price, confirm commitments, avoid unnecessary concessions.",
}
SUBMIT_SCHEMA = {"type": "object", "required": ["headline", "situation", "risk_signals", "recommended_actions", "price_guidance", "talking_points", "confidence"], "properties": {
  "headline": {"type": "string", "description": "One-sentence takeaway for the rep."},
  "situation": {"type": "string", "description": "2-3 sentences: where the deal stands and why."},
  "risk_signals": {"type": "array", "items": {"type": "object", "required": ["signal", "evidence"], "properties": {"signal": {"type": "string"}, "evidence": {"type": "array", "items": {"type": "string"}, "description": "note ids (e.g. NOTE0002629) or one of: model, interactions, similar_deals, account_history, portfolio"}}}},
  "recommended_actions": {"type": "array", "minItems": 1, "maxItems": 5, "items": {"type": "object", "required": ["priority", "play", "action", "why", "evidence", "timing"], "properties": {
      "priority": {"type": "integer"}, "play": {"type": "string", "enum": list(PLAYBOOK)}, "action": {"type": "string"}, "why": {"type": "string"},
      "evidence": {"type": "array", "items": {"type": "string"}}, "timing": {"type": "string"}}}},
  "price_guidance": {"type": "object", "required": ["stance", "detail"], "properties": {"stance": {"type": "string", "enum": ["hold price", "small concession may help", "do not discount"]},
      "detail": {"type": "string"}, "max_concession_pct": {"type": "number", "description": "0 to 5"}}},
  "talking_points": {"type": "array", "items": {"type": "string"}, "maxItems": 4},
  "questions_to_ask": {"type": "array", "items": {"type": "string"}, "maxItems": 3},
  "confidence": {"type": "string", "enum": ["low", "medium", "high"]}, "data_gaps": {"type": "array", "items": {"type": "string"}}}}

TOOL_CONFIG = {"tools": [
 {"toolSpec": {"name": "get_deal_brief", "description": "Model win probability (at offer time), drivers, price what-if, price vs competitor, account profile.", "inputSchema": {"json": {"type": "object", "properties": {"opportunity_id": {"type": "string"}}, "required": ["opportunity_id"]}}}},
 {"toolSpec": {"name": "get_recent_interactions", "description": "Contact cadence, issue flags and latest interactions for the account in the last 12 months.", "inputSchema": {"json": {"type": "object", "properties": {"opportunity_id": {"type": "string"}}, "required": ["opportunity_id"]}}}},
 {"toolSpec": {"name": "search_account_notes", "description": "Semantic search over this account's call notes (product-relevant, last 12 months). Use specific queries; call it more than once for different topics.", "inputSchema": {"json": {"type": "object", "properties": {"opportunity_id": {"type": "string"}, "query": {"type": "string"}, "k": {"type": "integer"}}, "required": ["opportunity_id", "query"]}}}},
 {"toolSpec": {"name": "get_account_history", "description": "This account's past deals (previous seasons): outcome, our price vs competitor, incumbent then, rounds, concession. Use as context, not as a forecast.", "inputSchema": {"json": {"type": "object", "properties": {"opportunity_id": {"type": "string"}}, "required": ["opportunity_id"]}}}},
 {"toolSpec": {"name": "get_portfolio_view", "description": "The same account's other open 2025-26 deals (probability, price position) - for bundling and consistent messaging.", "inputSchema": {"json": {"type": "object", "properties": {"opportunity_id": {"type": "string"}}, "required": ["opportunity_id"]}}}},
 {"toolSpec": {"name": "get_similar_past_deals", "description": "Win rate of comparable past deals (same incumbency / customer type / price position).", "inputSchema": {"json": {"type": "object", "properties": {"opportunity_id": {"type": "string"}}, "required": ["opportunity_id"]}}}},
 {"toolSpec": {"name": "submit_recommendation", "description": "Submit the final win plan. Call exactly once, when you have enough evidence.", "inputSchema": {"json": SUBMIT_SCHEMA}}},
]}

SYSTEM_PROMPT = '''You are a commercial coach for Aurevia Pharmaceutical field reps (flu vaccines: Vaxora HD, Vaxora Standard, Immunexa).
Goal: give the rep a short, specific plan that raises the chance of winning this contract without giving away margin unnecessarily.

How to work
1. Call get_deal_brief, get_recent_interactions, get_account_history, get_portfolio_view and get_similar_past_deals first, then search_account_notes at least twice on DIFFERENT topics (e.g. competitor / price / switching; service / supply; clinical preference; expansion).
2. Use ONLY facts returned by the tools. Never invent customers, numbers, quotes or events. If evidence is thin, say so in data_gaps and lower confidence.
3. Cite evidence: note ids for note-based claims; 'model', 'interactions', 'similar_deals', 'account_history' or 'portfolio' for the others.
4. Choose actions only from this playbook (use the play id):
''' + "\n".join(f"   - {k}: {v}" for k, v in PLAYBOOK.items()) + '''
5. Price guidance is advisory: never suggest a change outside -5%..+5% of the current offer; a person approves every price. The what-if is historical association, not proven cause.
   Prefer non-price levers when the account is not price-responsive (model gain from a 5% cut is small) or when the notes show service or clinical issues.
6. Account history and concession/round figures are CONTEXT, not forecasts (an account's past concession does not predict its next one; products at one account are not statistically linked). The model probability is an offer-time estimate; newer notes can override it - say so when they do.
7. Be concise and practical: at most 5 actions, ordered by priority, each with timing. No personal names. Finish by calling submit_recommendation.'''
print("playbook plays:", list(PLAYBOOK))

playbook plays: ['EXEC_SPONSOR', 'PRICE_VALUE_DEFENCE', 'TARGETED_CONCESSION', 'CLINICAL_EDUCATION', 'SERVICE_RECOVERY', 'PORTFOLIO_BUNDLE', 'COMPETITOR_COUNTER', 'ENGAGEMENT_BOOST', 'EXPANSION_PLAY', 'HOLD_AND_PROTECT']


## 5. Offline stand-in for Claude (used only when `DRY_RUN = True`)
A tiny rule-based "coach" that speaks the same Bedrock Converse protocol (tool calls → final submit). It lets you test the wiring, QA checks and layout without AWS, and shows what the LLM must improve on.

In [8]:
SIGNALS = [   # (signal, regex, plays)  - ordered by severity
 ("competitor re-tender / switch threat", r"re-?tender|rfp|seriously evaluating|evaluate alternatives|leaning toward|supplier review|shift \d+% of volume", ["EXEC_SPONSOR", "COMPETITOR_COUNTER"]),
 ("price pressure", r"budget pressure|cost containment|sharpen our pencil|price gap|reduction would be needed|come within \d+%|compared line items|cost per patient|lower on their latest offer|price adjustment|counter came in \d+% below|compared our offer", ["PRICE_VALUE_DEFENCE"]),
 ("service / supply issue", r"complaint|billing errors|below expectation|response times|delivery delay|emergency stock|disrespected", ["SERVICE_RECOVERY"]),
 ("competitor courting the account", r"visiting weekly|bundled deal|presented at their last|more responsive lately|pushing hard", ["COMPETITOR_COUNTER", "ENGAGEMENT_BOOST"]),
 ("clinical advocacy to build on", r"clinical committee session|prefers .{0,40} clinically|spoke positively|clinical preference will weigh|positive feedback|enhanced immunogenicity|efficacy in their high-risk", ["CLINICAL_EDUCATION"]),
 ("growth / expansion interest", r"expanding into|volume growth|affiliated sites", ["EXPANSION_PLAY"]),
]
def rule_based_plan(brief, inter, hits, similar, hist, port):
    m = brief["model_estimate_at_offer_time"]; p, gain = m["win_probability"], m["gain_from_5pct_price_cut"]
    found = {}
    for h in hits:
        for name, rx, plays in SIGNALS:
            if re.search(rx, h["text"].lower()): found.setdefault(name, {"plays": plays, "ids": []})["ids"].append(h["note_id"])
    same = hist.get("same_product_record", [])
    if same and same[-1]["outcome"] == "LOST":
        found["lost this product here last time (account history)"] = {"plays": ["COMPETITOR_COUNTER"], "ids": ["account_history"]}
    actions, used = [], set()
    for name, v in found.items():
        for pl in v["plays"]:
            if pl in used or len(actions) >= 4: continue
            used.add(pl); actions.append(dict(priority=len(actions) + 1, play=pl, action=PLAYBOOK[pl], why=f"Notes show: {name}.",
                                              evidence=list(dict.fromkeys(v["ids"]))[:2], timing="within 2 weeks"))
    if port.get("n_other_open_deals", 0) >= 1 and "PORTFOLIO_BUNDLE" not in used and len(actions) < 4:
        used.add("PORTFOLIO_BUNDLE"); actions.append(dict(priority=len(actions) + 1, play="PORTFOLIO_BUNDLE", action=PLAYBOOK["PORTFOLIO_BUNDLE"],
                      why=f"{port['n_other_open_deals']} other open deal(s) at this account.", evidence=["portfolio"], timing="before the next offer round"))
    if inter.get("days_since_last_contact", 0) > 60 and "ENGAGEMENT_BOOST" not in used:
        actions.append(dict(priority=len(actions) + 1, play="ENGAGEMENT_BOOST", action=PLAYBOOK["ENGAGEMENT_BOOST"], why=f"No contact for {inter['days_since_last_contact']} days.", evidence=["interactions"], timing="this week"))
    if not actions: actions.append(dict(priority=1, play="HOLD_AND_PROTECT" if p >= 0.5 else "ENGAGEMENT_BOOST", action="Keep the current plan and confirm next steps with the buyer.", why="No strong signals in recent notes.", evidence=["model"], timing="next meeting"))
    price_resp = gain is not None and gain >= 0.06 and p < 0.5
    stance = "small concession may help" if price_resp else ("hold price" if p >= 0.5 else "do not discount")
    return dict(headline=f"{brief['customer']} - {brief['product']}: {p:.0%} win probability ({m['risk_tier']}). " + ("; ".join(found) or "no strong note signals") + ".",
                situation=f"Offer is {brief['price_vs_competitor_pct']:+.1f}% vs competitor; incumbent: {brief['incumbent_supplier']}. Similar past deals won {similar['win_rate']:.0%} (n={similar['n_past_deals']}).",
                risk_signals=[dict(signal=k, evidence=list(dict.fromkeys(v["ids"]))[:3]) for k, v in found.items()] or [dict(signal="no strong risk signal in notes", evidence=["interactions"])],
                recommended_actions=actions[:5],
                price_guidance=dict(stance=stance, detail=f"Model: a 5% cut changes win probability by about {gain:+.1%} (association only).", max_concession_pct=2.5 if price_resp else 0),
                talking_points=["Lead with total value: supply reliability, service, clinical evidence."], questions_to_ask=["What would make this offer a clear yes for your team?"],
                confidence="medium" if found else "low", data_gaps=[] if found else ["No decisive notes found"])

class MockConverse:
    def converse(self, **kw):
        msgs = kw["messages"]; n_assistant = sum(m["role"] == "assistant" for m in msgs)
        oid = re.search(r"opportunity_id=(\S+)", msgs[0]["content"][0]["text"]).group(1).strip(".,")
        if n_assistant == 0:
            blocks = [{"toolUse": {"toolUseId": f"t{i}", "name": n, "input": a}} for i, (n, a) in enumerate([
                ("get_deal_brief", {"opportunity_id": oid}), ("get_recent_interactions", {"opportunity_id": oid}), ("get_similar_past_deals", {"opportunity_id": oid}),
                ("get_account_history", {"opportunity_id": oid}), ("get_portfolio_view", {"opportunity_id": oid}),
                ("search_account_notes", {"opportunity_id": oid, "query": "competitor price pressure switch re-tender RFP"}),
                ("search_account_notes", {"opportunity_id": oid, "query": "service complaint supply delivery clinical preference expansion"})])]
            return {"output": {"message": {"role": "assistant", "content": blocks}}, "stopReason": "tool_use", "usage": {"inputTokens": 0, "outputTokens": 0}}
        res = [b["toolResult"]["content"][0]["json"] for b in msgs[-1]["content"]]
        brief, inter, similar, hist, port = res[0], res[1], res[2], res[3], res[4]; hits = res[5].get("notes", []) + res[6].get("notes", [])
        return {"output": {"message": {"role": "assistant", "content": [{"toolUse": {"toolUseId": "final", "name": "submit_recommendation", "input": rule_based_plan(brief, inter, hits, similar, hist, port)}}]}},
                "stopReason": "tool_use", "usage": {"inputTokens": 0, "outputTokens": 0}}
print("mock ready" if DRY_RUN else "(mock not used in live mode)")

mock ready


## 6. The agent loop and quality checks
* Claude decides which tools to call (parallel tool calls are handled), reads the results, and finishes by calling `submit_recommendation`.
* **QA checks** run on every answer: schema complete, plays exist in the playbook, every cited note id was actually retrieved, price guidance within ±5 %, no leftover personal names. Failures are shown with the plan, not hidden.

In [9]:
def run_agent(opportunity_id, user_question=None, max_steps=8, verbose=True):
    client = MockConverse() if DRY_RUN else bedrock()
    focus = f"\nThe rep's question: {user_question}" if user_question else ""
    messages = [{"role": "user", "content": [{"text": f"Prepare the win plan for opportunity_id={opportunity_id}.{focus}"}]}]
    seen_ids, trace, tokens = set(), [], [0, 0]
    for step in range(max_steps):
        resp = client.converse(modelId=LLM_MODEL_ID, system=[{"text": SYSTEM_PROMPT}], messages=messages, toolConfig=TOOL_CONFIG,
                               inferenceConfig={"maxTokens": 3000, "temperature": 0.2})
        tokens[0] += resp.get("usage", {}).get("inputTokens", 0); tokens[1] += resp.get("usage", {}).get("outputTokens", 0)
        msg = resp["output"]["message"]; messages.append(msg)
        uses = [b["toolUse"] for b in msg["content"] if "toolUse" in b]
        if not uses:   # model answered in text instead of calling the submit tool -> nudge once
            messages.append({"role": "user", "content": [{"text": "Please finish by calling submit_recommendation with your plan."}]}); continue
        results, final = [], None
        for u in uses:
            if u["name"] == "submit_recommendation": final = u["input"]; continue
            try:
                args = dict(u["input"]); args["opportunity_id"] = opportunity_id            # the agent can only look at THIS deal
                out = jsonable(TOOLS[u["name"]](**args)); status = "success"
                seen_ids |= {n["note_id"] for n in out.get("notes", [])}
            except Exception as e:
                out, status = {"error": f"{type(e).__name__}: {e}"}, "error"
            trace.append((u["name"], {k: v for k, v in u["input"].items() if k != "opportunity_id"}, status))
            results.append({"toolResult": {"toolUseId": u["toolUseId"], "content": [{"json": out}], "status": status}})
        if final is not None:
            return finalize(opportunity_id, final, seen_ids, trace, tokens, step + 1, verbose)
        messages.append({"role": "user", "content": results})
    raise RuntimeError("Agent did not submit a recommendation within the step limit")

def qa_checks(rec, seen_ids):
    issues = []
    for k in SUBMIT_SCHEMA["required"]:
        if k not in rec: issues.append(f"missing field: {k}")
    ok_tokens = seen_ids | {"model", "interactions", "similar_deals", "account_history", "portfolio"}
    for a in rec.get("recommended_actions", []):
        if a.get("play") not in PLAYBOOK: issues.append(f"play not in playbook: {a.get('play')}")
        bad = [e for e in a.get("evidence", []) if e not in ok_tokens]
        if bad: issues.append(f"unsupported evidence in action {a.get('priority')}: {bad}")
        if not a.get("evidence"): issues.append(f"action {a.get('priority')} has no evidence")
    for s in rec.get("risk_signals", []):
        bad = [e for e in s.get("evidence", []) if e not in ok_tokens]
        if bad: issues.append(f"unsupported evidence in risk signal '{s.get('signal')}': {bad}")
    pg = rec.get("price_guidance", {})
    if abs(pg.get("max_concession_pct", 0) or 0) > 5: issues.append("price guidance outside +/-5% range")
    if re.search(r"Dr\.\s|Michelle|Sarah", json.dumps(rec)): issues.append("possible personal name in output")
    return issues

def finalize(opportunity_id, rec, seen_ids, trace, tokens, steps, verbose):
    issues = qa_checks(rec, seen_ids)
    if verbose: print(f"agent finished in {steps} step(s) | tools: {[t[0] + (str(t[1]) if t[1] else '') for t in trace]} | tokens in/out: {tokens[0]}/{tokens[1]} | QA issues: {len(issues)}")
    return {"opportunity_id": opportunity_id, "plan": rec, "qa_issues": issues, "tool_trace": trace, "notes_seen": sorted(seen_ids)}

## 7. One-page brief for the rep

In [10]:
def render(result):
    r, p, d = result["plan"], result["plan"], deals.loc[result["opportunity_id"]]
    md = [f"## {d.customer_name} — {d.product_name}  ·  win probability **{d.win_probability:.0%}** ({d.risk_tier})",
          f"*{d.customer_type} · {d.region} · incumbent: {d.incumbent_supplier} · our price {d.price_vs_comp_pct:+.1f}% vs competitor · rep {d.rep_id} · data as of {AS_OF:%Y-%m-%d}*", "",
          f"**{r['headline']}**", "", r["situation"], "", "### Risk signals"]
    md += [f"- {s['signal']}  `{', '.join(s['evidence'])}`" for s in r["risk_signals"]]
    md += ["", "### Recommended actions"]
    md += [f"{a['priority']}. **{a['play'].replace('_', ' ').title()}** — {a['action']}  \n   *Why:* {a['why']} · *When:* {a['timing']} · `{', '.join(a['evidence'])}`" for a in sorted(r["recommended_actions"], key=lambda x: x["priority"])]
    pg = r["price_guidance"]
    md += ["", f"### Price guidance (advisory — a manager approves every price)", f"**{pg['stance'].capitalize()}**" + (f" · up to {pg.get('max_concession_pct', 0):.1f}%" if pg.get("max_concession_pct") else "") + f" — {pg['detail']}"]
    if r.get("talking_points"): md += ["", "### Talking points"] + [f"- {t}" for t in r["talking_points"]]
    if r.get("questions_to_ask"): md += ["", "### Questions to ask"] + [f"- {t}" for t in r["questions_to_ask"]]
    md += ["", f"*Confidence: {r['confidence']}*" + (f" · data gaps: {'; '.join(r['data_gaps'])}" if r.get("data_gaps") else "")]
    if result["qa_issues"]: md += ["", "⚠️ **QA flags — review before using:** " + "; ".join(result["qa_issues"])]
    evidence = [h for h in INDEX.df[INDEX.df.note_id.isin(result["notes_seen"])].sort_values("interaction_date").itertuples()]
    md += ["", "<details><summary>Notes the agent saw</summary>", ""] + [f"- `{h.note_id}` {h.interaction_date:%Y-%m-%d} ({h.interaction_type}): {h.text}" for h in evidence] + ["", "</details>"]
    display(Markdown("\n".join(md)))
    hist_t = get_account_history(result["opportunity_id"]); port_t = get_portfolio_view(result["opportunity_id"])
    if hist_t.get("past_deals_latest"):
        display(Markdown("**Account history (structured data)**")); display(pd.DataFrame(hist_t["past_deals_latest"]).round(1))
    if port_t["other_open_deals"]:
        display(Markdown("**Other open deals at this account**")); display(pd.DataFrame(port_t["other_open_deals"]).round(3))

## 8. Test connection (live mode only)
Run once. If this fails, the error text tells you what to fix (IAM permission for `bedrock:InvokeModel` / `Converse` on the inference profile and Titan model, or region).

In [11]:
if not DRY_RUN:
    try:
        r = bedrock().converse(modelId=LLM_MODEL_ID, messages=[{"role": "user", "content": [{"text": "Reply with the single word: ready"}]}], inferenceConfig={"maxTokens": 20})
        print("Claude:", r["output"]["message"]["content"][0]["text"], "| embeddings dim:", len(titan_embed("test")))
    except Exception as e:
        print("Bedrock check FAILED ->", type(e).__name__, str(e)[:400])
else:
    print("DRY_RUN = True: skipping Bedrock check")

DRY_RUN = True: skipping Bedrock check


## 9. Where to look first — priority list (no LLM calls)
Rough "value at risk" index = (1 − win probability) × estimated annual doses × our price. It is a **proxy** (true volume and margin are not in the data); use it only to decide which deals to ask about.

In [12]:
pri = deals.assign(value_at_risk=(1 - deals.win_probability) * deals.annual_flu_doses_est * deals.sim_offer_price)
view = pri.sort_values("value_at_risk", ascending=False).head(12)[["opportunity_id", "customer_name", "product_name", "rep_id", "win_probability", "risk_tier", "gain_from_5pct_cut", "recommendation"]]
view.round(3).reset_index(drop=True)

,opportunity_id,customer_name,product_name,rep_id,win_probability,risk_tier,gain_from_5pct_cut,recommendation
0,OPP001215,MediSource Alliance,Immunexa,REP014,0.488,Contested,0.089,Price-responsive: a discount can move the odds
1,OPP001426,VaccineDirect Consortium,Vaxora HD,REP034,0.231,At risk,0.066,Long shot: use non-price levers / executive sp...
2,OPP001131,VaccineDirect Consortium,Immunexa,REP008,0.380,Contested,0.075,Price-responsive: a discount can move the odds
3,OPP001480,MediSource Alliance,Immunexa,REP015,0.306,Contested,0.073,Price-responsive: a discount can move the odds
4,OPP001197,NationalCare Purchasing,Immunexa,REP001,0.362,Contested,0.076,Price-responsive: a discount can move the odds
5,OPP001216,MediSource Alliance,Vaxora HD,REP014,0.477,Contested,0.073,Price-responsive: a discount can move the odds
6,OPP001154,VaccineDirect Consortium,Immunexa,REP023,0.330,Contested,0.075,Price-responsive: a discount can move the odds
7,OPP001288,HealthCoop National,Immunexa,REP003,0.204,At risk,0.064,Long shot: use non-price levers / executive sp...
8,OPP001324,HealthCoop National,Immunexa,REP006,0.186,At risk,0.060,Long shot: use non-price levers / executive sp...
9,OPP001437,NationalCare Purchasing,Immunexa,REP029,0.456,Contested,0.073,Price-responsive: a discount can move the odds


# 10. ✍️ Ask your question
Type a question or an opportunity id, then run the cell. Examples:
* `"OPP001150"` — just the opportunity id
* `"What should I do for Riverside Health System on Immunexa? They keep mentioning a competitor."` — the customer name (and product, if you mention it) is matched automatically
* `"How do I save OPP001150 without cutting the price?"`

`REP_ID` is optional: set it (e.g. `"REP007"`) to restrict the name search to that rep's accounts.

In [13]:
QUESTION = "OPP001113"        # <-- type your question here
REP_ID   = None               # <-- optional, e.g. "REP007"

def resolve_opportunity(question, rep_id=None):
    m = re.search(r"OPP\d+", question.upper())
    if m and m.group(0) in deals.index: return m.group(0)
    pool = deals if rep_id is None else deals[deals.rep_id == rep_id]
    names = pool.customer_name.str.lower()
    q = question.lower()
    hits = [n for n in names.unique() if n in q]
    if not hits:                                   # fuzzy fallback on the closest customer name
        close = difflib.get_close_matches(q, list(names.unique()), n=1, cutoff=0.0)
        hits = close
    cand = pool[names == hits[0]]
    prod = [p for p in cand.product_name.unique() if p.lower() in q]
    if prod: cand = cand[cand.product_name == prod[0]]
    if len(cand) > 1: print("Several matching deals - using the lowest win probability. Matches:", list(cand.opportunity_id))
    return cand.sort_values("win_probability").opportunity_id.iloc[0]

opp = resolve_opportunity(QUESTION, REP_ID)
print("Resolved to:", opp, "|", deals.loc[opp, "customer_name"], "-", deals.loc[opp, "product_name"])
result = run_agent(opp, user_question=None if re.fullmatch(r"\s*OPP\d+\s*", QUESTION, re.I) else QUESTION)
render(result)

Resolved to: OPP001113 | Harbor Riverside Medical Center - Vaxora Standard
agent finished in 2 step(s) | tools: ['get_deal_brief', 'get_recent_interactions', 'get_similar_past_deals', 'get_account_history', 'get_portfolio_view', "search_account_notes{'query': 'competitor price pressure switch re-tender RFP'}", "search_account_notes{'query': 'service complaint supply delivery clinical preference expansion'}"] | tokens in/out: 0/0 | QA issues: 0


## Harbor Riverside Medical Center — Vaxora Standard  ·  win probability **29%** (Contested)
*Health System · Northeast · incumbent: AeroFlu · our price +0.0% vs competitor · rep REP002 · data as of 2025-12-25*

**Harbor Riverside Medical Center - Vaxora Standard: 29% win probability (Contested). competitor courting the account; clinical advocacy to build on; growth / expansion interest; lost this product here last time (account history).**

Offer is +0.0% vs competitor; incumbent: AeroFlu. Similar past deals won 27% (n=41).

### Risk signals
- competitor courting the account  `NOTE0002629`
- clinical advocacy to build on  `NOTE0002631`
- growth / expansion interest  `NOTE0002631`
- lost this product here last time (account history)  `account_history`

### Recommended actions
1. **Competitor Counter** — Prepare a factual comparison against the named competitor and address its specific claims or offers.  
   *Why:* Notes show: competitor courting the account. · *When:* within 2 weeks · `NOTE0002629`
2. **Engagement Boost** — Raise contact cadence and add stakeholders (pharmacy, clinical committee, procurement) - multi-thread the account.  
   *Why:* Notes show: competitor courting the account. · *When:* within 2 weeks · `NOTE0002629`
3. **Clinical Education** — Schedule a clinical-committee session / data review (e.g. immunogenicity, high-risk populations) to reinforce clinical preference.  
   *Why:* Notes show: clinical advocacy to build on. · *When:* within 2 weeks · `NOTE0002631`
4. **Expansion Play** — Account signals growth: propose expanded volume / additional sites on current terms.  
   *Why:* Notes show: growth / expansion interest. · *When:* within 2 weeks · `NOTE0002631`

### Price guidance (advisory — a manager approves every price)
**Small concession may help** · up to 2.5% — Model: a 5% cut changes win probability by about +7.0% (association only).

### Talking points
- Lead with total value: supply reliability, service, clinical evidence.

### Questions to ask
- What would make this offer a clear yes for your team?

*Confidence: medium*

<details><summary>Notes the agent saw</summary>

- `NOTE0002629` 2025-05-19 (virtual_meeting): Heard through the [CONTACT] channel that AeroFlu has been visiting weekly. They're pushing hard on their AeroFlu. We need to respond with more than price.
- `NOTE0002631` 2025-07-25 (conference_meeting): Great meeting with [CONTACT] in supply chain. Their team highlighted our portfolio's efficacy in their high-risk population. Wants to explore expanding into affiliated sites.

</details>

**Account history (structured data)**

,season,product,outcome,our_price_vs_competitor_pct,incumbent_then,competitor,rounds,concession_pct
0,2022-23,Vaxora Standard,LOST,1.3,MediShield,MediShield,1,None
1,2023-24,Vaxora HD,LOST,7.3,Other,MediShield Plus,2,None
2,2024-25,Vaxora HD,LOST,3.3,Other,MediShield Plus,1,None
